# UD4: Visión Artificial Avanzada - DeepFace & MediaPipe 🧠👁️

¡Bienvenidos de nuevo! En este cuaderno vamos a sumergirnos en el mundo del **Análisis Facial Profundo**. 

Hasta ahora hemos detectado objetos y caras, pero... ¿podemos entender *quién* es esa persona? ¿O *cómo* se siente? ¿O *adónde* está mirando?

Usaremos dos herramientas superpotentes:
1.  **DeepFace**: Un framework de reconocimiento facial híbrido que envuelve modelos de vanguardia como VGG-Face, Google FaceNet, OpenFace, etc. Es increíblemente fácil de usar.
2.  **MediaPipe (Iris)**: Para un seguimiento de ojos y pupilas de altísima precisión.

---
## 0. Configuración del Entorno 🛠️

Esta práctica requiere librerías pesadas. Asegúrate de tener instalado todo. 
Nota: `tf-keras` es necesario para que DeepFace funcione correctamente con las versiones modernas de TensorFlow.

`PIA_DeepFace_310_v2`

In [5]:
# Instalación de dependencias (ejecuta si te faltan)
# vamos a añadir el instalador de UV, que busca imcompatibilidades entre librerías
!pip install uv 
!uv pip install deepface tf-keras opencv-python pandas matplotlib

Using Python 3.10.19 environment at: C:\Users\adria\anaconda3\envs\PIA_Vision_310
Resolved 76 packages in 1.17s
error: failed to remove file `C:\Users\adria\anaconda3\envs\PIA_Vision_310\Lib\site-packages\google/protobuf/internal/_api_implementation.cp310-win_amd64.pyd`: Acceso denegado. (os error 5)


In [6]:
import cv2
from deepface import DeepFace
import numpy as np
import matplotlib.pyplot as plt
import os

# Configuración para mostrar imágenes en el notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 6]

print("¡Librerías importadas correctamente! 🚀")

ModuleNotFoundError: No module named 'deepface'

---
## 1. Análisis de Atributos en Tiempo Real 🎭

DeepFace no solo reconoce caras, también puede analizar atributos:
- **Emoción**: Enojado, Miedo, Neutral, Triste, Disgusto, Feliz, Sorpresa.
- **Edad**: Una estimación numérica.
- **Género**: Mujer / Hombre.
- **Raza**: Asiático, Indio, Negro, Blanco, Hispano, etc.

Vamos a usar `DeepFace.analyze()`. Esta función es "mágica" pero pesada. En tiempo real, no podemos llamarla en cada frame o el vídeo irá a 1 FPS. 

**Estrategia**: Analizar solo cada X frames (por ejemplo, cada 30 frames).

In [2]:
# INICIALIZAR LA CÁMARA
captura = cv2.VideoCapture(0)

atributos_actuales = "Analizando..."
contador_frames = 0

while True:
    exito, imagen = captura.read()
    if not exito: break
    
    # Espejo para que sea más natural
    imagen = cv2.flip(imagen, 1)
    copia_imagen = imagen.copy()
    
    # Analizamos cada 30 frames para no congelar la CPU
    if contador_frames % 30 == 0:
        try:
            # actions=['emotion', 'age', 'gender'] 
            # enforce_detection=False para que no crashee si no ve cara
            # silent=True para que no muestre mensajes irrelevantes en consola
            resultados = DeepFace.analyze(copia_imagen, actions=['emotion', 'age', 'gender'], enforce_detection=False, silent=True) 
            
            if resultados:
                # DeepFace devuelve una lista de diccionarios (uno por cara)
                primer_resultado = resultados[0]
                emocion = primer_resultado['dominant_emotion']
                edad = primer_resultado['age']
                genero = primer_resultado['dominant_gender']
                
                atributos_actuales = f"{emocion.upper()} | {edad} anios | {genero}"
        except Exception as e:
            print(f"Error en analisis: {e}")
    
    # Dibujar resultados (incluso si no analizamos en este frame, mostramos el último resultado)
    cv2.rectangle(imagen, (10, 10), (600, 60), (0, 0, 0), -1)
    cv2.putText(imagen, atributos_actuales, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    cv2.imshow('Analisis de Atributos DeepFace', imagen)
    contador_frames += 1
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

captura.release()
cv2.destroyAllWindows()

### 🧠 Examen de Mentira 1: El Espejo Inteligente 🪞

**Objetivo**: Modifica el código anterior para crear un "Espejo Amable".
1. Si detecta que estás **"sad"** (triste), debe poner un texto en pantalla: *"¡Animate! Todo va a salir bien"* en color VERDE.
2. Si detecta que estás **"happy"** (feliz), debe poner: *"¡Esa es la actitud!"* en color AMARILLO.
3. Si detecta que estás **"angry"** (enfadado), debe poner: *"Relájate, respira hondo"* en color AZUL.
4. Para cualquier otra emoción, muestra simplemente la emoción.

Hazlo en la siguiente celda.

In [ ]:
# TODO: Tu código del Espejo Inteligente aquí



---
## 2. Verificación Facial (Login Biométrico) 🔐

La verificación responde a la pregunta: **¿Es esta persona quien dice ser?**
Comparamos una cara 1 (Webcam) contra una cara 2 (Base de datos).

Usaremos dos imágenes de prueba que hemos generado en `img/sujeto_1.png` y `img/sujeto_2.png`. 
- Imagina que `sujeto_1.png` es el usuario **AUTORIZADO**.
- Cualquier otra persona es un **INTRUSO**.

DeepFace calcula la *distancia* entre los vectores de las caras. Si la distancia es menor que un umbral, son la misma persona.

In [ ]:
# Crear la carpeta img si no existe para evitar errores
if not os.path.exists("img"):
    os.makedirs("img")

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("¡ERROR! No se puede acceder a la webcam.")
else:
    print("Presiona 'c' para capturar y guardar, o 'q' para salir...")
    
    while True:
        exito, imagen = cap.read()
        if not exito: break

        cv2.imshow('Captura de Usuario', imagen)

        key = cv2.waitKey(1)
        if key & 0xFF == ord('c'):
            # --- GUARDAR LA IMAGEN ---
            cv2.imwrite("img/sujeto_1.png", imagen)
            img_autorizado = imagen 
            print("Imagen guardada en img/sujeto_1.png")
            break
        elif key & 0xFF == ord('q'):
            img_autorizado = None
            break

    cap.release()
    cv2.destroyAllWindows()

    if img_autorizado is not None:
        plt.imshow(cv2.cvtColor(img_autorizado, cv2.COLOR_BGR2RGB))
        plt.title("Sujeto 1 Guardado")
        plt.show()

In [4]:
# SISTEMA DE LOGIN BIOMÉTRICO

captura = cv2.VideoCapture(0)

mensaje_estado = "Esperando verificacion..."
color_estado = (255, 255, 255)
ruta_autorizado = "img\sujeto_1.png"

while True:
    exito, imagen = captura.read()
    if not exito: break
    imagen = cv2.flip(imagen, 1)
    
    # Verificamos al pulsar 'v' para no saturar
    tecla = cv2.waitKey(1) & 0xFF
    
    if tecla == ord('v'):
        mensaje_estado = "Verificando..."
        try:
            # DeepFace.verify compara dos imagenes
            # model_name='VGG-Face' es rápido y bueno
            verificacion = DeepFace.verify(img1_path = imagen, img2_path = ruta_autorizado, model_name='VGG-Face', enforce_detection=False)
            
            es_valido = verificacion['verified']
            distancia = verificacion['distance']
            
            if es_valido:
                mensaje_estado = f"ACCESO CONCEDIDO (Dist: {distancia:.2f})"
                color_estado = (0, 255, 0) # Verde
            else:
                mensaje_estado = f"ACCESO DENEGADO (Dist: {distancia:.2f})"
                color_estado = (0, 0, 255) # Rojo
                
        except Exception as e:
            print(e)
            mensaje_estado = "Error en verificacion"

    # Interfaz
    cv2.putText(imagen, "Pulsa 'v' para verificar", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    cv2.rectangle(imagen, (0, 400), (640, 480), (0, 0, 0), -1)
    cv2.putText(imagen, mensaje_estado, (50, 450), cv2.FONT_HERSHEY_SIMPLEX, 1, color_estado, 2)
    
    cv2.imshow('Login Biometrico', imagen)
    
    if tecla == ord('q'):
        break

captura.release()
cv2.destroyAllWindows()

### 🧠 Examen de Mentira 2: El Portero de Discoteca 🚫

**Objetivo**: Implementa un sistema que verifique contra DOS personas autorizadas (puedes usar `sujeto_1.png` y `sujeto_2.png`).
1. Carga ambas imágenes.
2. Cuando pulses 'v', verifica el frame actual contra el Sujeto 1.
3. Si es falso, verifica contra el Sujeto 2.
4. Si alguno da `True`, muestra "Pase VIP". Si ninguno da `True`, muestra "No estás en la lista".

*Pista*: Usa un `if ... elif ... else`.

In [ ]:
# TODO: Tu código del Portero de Discoteca aquí



---
## 3. Vectores Numéricos (Embeddings) en Tiempo Real 🔢

Las redes neuronales no ven "caras", ven números. Una cara se convierte en un vector (una lista de números, por ejemplo 128 o 2624 números).

Vamos a visualizar esos números en tiempo real. Es lo que la IA "ve" realmente.
Usaremos `DeepFace.represent()`. 

OJO: sólo la máquina entiende el significado de los números pero podrían ser factores como: profundidad de las cuencas oculares, forma de la mandíbula, etc.

In [5]:
captura = cv2.VideoCapture(0)
contador_frames = 0
vector_str = ""

while True:
    exito, imagen = captura.read()
    if not exito: break
    imagen = cv2.flip(imagen, 1)
    
    # Representamos cada pocos frames
    if contador_frames % 20 == 0:
        try:
            # Obtener el embedding (vector)
            embedding_objs = DeepFace.represent(imagen, model_name='Facenet', enforce_detection=False)
            embedding = embedding_objs[0]['embedding']
            
            # Tomamos solo los primeros 5 números para mostrar, el vector completo es enorme
            vector_corto = [round(num, 2) for num in embedding[:5]]
            vector_str = f"Vector: {vector_corto}..."
        except:
            pass
            
    cv2.putText(imagen, vector_str, (20, 450), cv2.FONT_HERSHEY_PLAIN, 1.5, (0, 255, 255), 2)
    cv2.imshow('Embeddings en Tiempo Real', imagen)
    contador_frames += 1
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

captura.release()
cv2.destroyAllWindows()

---
## 4. Detección y Alineación 📐

Antes de reconocer a alguien, hay que encontrar su cara y "enderezarla" (alineación). DeepFace hace esto automáticamente, pero podemos visualizar la cara detectada.


In [6]:
import cv2
from deepface import DeepFace

captura = cv2.VideoCapture(0)
contador_frames = 0  # Inicializamos el contador

while True:
    exito, imagen = captura.read()
    if not exito: break
    imagen = cv2.flip(imagen, 1)
    
    # Solo procesamos la IA cada 30 frames para evitar que el PC se bloquee
    if contador_frames % 30 == 0:
        try:
            # 'retinaface' es muy preciso pero lento; 'align=True' endereza el rostro
            caras = DeepFace.extract_faces(imagen, 
                                        detector_backend='retinaface', 
                                        enforce_detection=False, 
                                        align=True) 

            for cara in caras:
                # Rostro ya recortado y alineado (ojos horizontales)
                rostro_procesado = cara['face'] 
                
                # Mostramos la cara "enderezada" en una ventana aparte
                cv2.imshow('Cara Alineada', rostro_procesado)
                
                # Guardamos el área para dibujar el rectángulo en el frame actual
                area = cara['facial_area']
                cv2.rectangle(imagen, (area['x'], area['y']), 
                            (area['x']+area['w'], area['y']+area['h']), (0, 255, 0), 2)
                            
        except Exception as e:
            # Si no hay cara o hay error, no hacemos nada
            pass
    
    # IMPORTANTE: Mostrar siempre la imagen original para que el video sea fluido
    cv2.imshow('Deteccion Facial', imagen)
    
    contador_frames += 1 # Incrementamos en cada ciclo
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

captura.release()
cv2.destroyAllWindows()